# Sistemas de Información Ambiental

> **Contexto de dominio:** [`docs/fuentes/sistemas_informacion_ambiental.md`](../../docs/fuentes/sistemas_informacion_ambiental.md)  
> **Bloque:** A | **Línea:** `sistemas_informacion_ambiental`  
> **Variable principal:** `n_registros` ()  
> **Modelos sugeridos:** RANDOM_FOREST, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/sistemas_informacion_ambiental.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "sistemas_informacion_ambiental"
VARIABLE = "n_registros"
UNIDAD = ""

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `sistemas_informacion_ambiental` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "sistemas_informacion_ambiental.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/sistemas_informacion_ambiental.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/sistemas_informacion_ambiental.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "n_registros": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `sistemas_informacion_ambiental` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_sistemas_informacion_ambiental.html",
       title="EDA — Sistemas de Información Ambiental", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "n_registros", title="Sistemas de Información Ambiental — n_registros ()")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "n_registros", period="month")
plt.show()

<!-- ENRICHMENT: sistemas_informacion_ambiental -->

## 3c. RETC — Cargas de vertimientos y metales pesados en biosólidos

El **RETC (Registro de Emisiones y Transferencia de Contaminantes)**, integrado al RUA (Res. 839/2023), cuantifica cargas contaminantes vertidas al agua por sector productivo:

| Parametro | Referencia | Umbral reporte |
|---|---|---|
| DBO (vertimientos) | Res. 631/2015 | Segun tipo de actividad |
| SST (vertimientos) | Res. 631/2015 | Segun tipo de actividad |
| Cadmio en biosolidos | Decreto 1287/2014 | < 39 mg/kg MS |
| Plomo en biosolidos | Decreto 1287/2014 | < 300 mg/kg MS |

Colombia vertio ~1.39 M ton DBO y ~1.35 M ton SST en 2020 (IDEAM/SIRH).

> **Tasa retributiva (Decreto 2667/2012):** cargo economico por kg DBO o SST vertido, incentivo economico para que los generadores reduzcan sus cargas.

In [ ]:
# RETC sintetico: cargas de vertimientos por sector + metales en biosolidos
np.random.seed(19)
n_anos = 10
anos = list(range(2014, 2014 + n_anos))
sectores_vert = ['Municipal', 'Agroindustria', 'Mineria', 'Manufactura', 'Servicios']

# Cargas DBO por sector (miles ton DBO/ano)
dbo_kt = {
    'Municipal':     [800 - i*5  + np.random.normal(0, 20) for i in range(n_anos)],
    'Agroindustria': [300 + i*2  + np.random.normal(0, 10) for i in range(n_anos)],
    'Mineria':       [120 - i*1  + np.random.normal(0, 5)  for i in range(n_anos)],
    'Manufactura':   [180 - i*3  + np.random.normal(0, 8)  for i in range(n_anos)],
    'Servicios':     [40  + i*0.5+ np.random.normal(0, 2)  for i in range(n_anos)],
}
df_retc = pd.DataFrame({'ano': anos,
                         **{s: [round(v,1) for v in vals]
                            for s, vals in dbo_kt.items()}})
df_retc['total_dbo_kt'] = df_retc[sectores_vert].sum(axis=1).round(1)

# Metales pesados en biosolidos PTAR (mg/kg MS) -- Decreto 1287/2014
n_ptar = 15
ptars = [f'PTAR-{i+1:02d}' for i in range(n_ptar)]
cadmio = np.random.uniform(10, 95, n_ptar).round(1)   # umbral: 39 mg/kg
plomo  = np.random.uniform(80, 950, n_ptar).round(1)  # umbral: 300 mg/kg
df_biosolidos = pd.DataFrame({'PTAR': ptars, 'Cadmio_mgkg': cadmio, 'Plomo_mgkg': plomo})
df_biosolidos['Cd_cumple'] = df_biosolidos['Cadmio_mgkg'] <= 39
df_biosolidos['Pb_cumple'] = df_biosolidos['Plomo_mgkg'] <= 300

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel A: Cargas DBO vertimientos por sector (stacked)
ax = axes[0]
bottom = np.zeros(n_anos)
colors_vert = ['#e74c3c','#e67e22','#f1c40f','#3498db','#9b59b6']
for s, color in zip(sectores_vert, colors_vert):
    ax.bar(df_retc['ano'], df_retc[s], bottom=bottom, label=s, color=color, alpha=0.85, width=0.7)
    bottom += df_retc[s].values
ax.set_title('RETC — Carga DBO vertida al agua (kt/ano)', fontweight='bold')
ax.set_ylabel('kt DBO/ano'); ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

# Panel B: Cadmio en biosolidos
ax = axes[1]
bar_cd = ['#27ae60' if v <= 39 else '#e74c3c' for v in cadmio]
ax.barh(ptars, cadmio, color=bar_cd, alpha=0.85)
ax.axvline(39, color='red', ls='--', lw=1.5, label='Umbral Cd: 39 mg/kg (Dto. 1287/2014)')
ax.set_title('Cadmio en Biosolidos PTAR', fontweight='bold')
ax.set_xlabel('Cadmio (mg/kg MS)'); ax.legend(fontsize=7); ax.grid(axis='x', alpha=0.3)

# Panel C: Plomo en biosolidos
ax = axes[2]
bar_pb = ['#27ae60' if v <= 300 else '#e74c3c' for v in plomo]
ax.barh(ptars, plomo, color=bar_pb, alpha=0.85)
ax.axvline(300, color='red', ls='--', lw=1.5, label='Umbral Pb: 300 mg/kg (Dto. 1287/2014)')
ax.set_title('Plomo en Biosolidos PTAR', fontweight='bold')
ax.set_xlabel('Plomo (mg/kg MS)'); ax.legend(fontsize=7); ax.grid(axis='x', alpha=0.3)

plt.suptitle('SIAC / RETC — Monitoreo de cargas contaminantes y biosolidos (Res. 839/2023)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

ptar_cd_incumple = (~df_biosolidos['Cd_cumple']).sum()
ptar_pb_incumple = (~df_biosolidos['Pb_cumple']).sum()
print(f'PTAR incumple Cadmio (>39 mg/kg): {ptar_cd_incumple}/{n_ptar}')
print(f'PTAR incumple Plomo (>300 mg/kg): {ptar_pb_incumple}/{n_ptar}')
print(f'Total vertimientos 2023 (estimado): {df_retc["total_dbo_kt"].iloc[-1]:.0f} kt DBO')

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["n_registros"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `n_registros` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["n_registros"], variable="n_registros")
if rep.empty:
    print("Sin normas colombianas registradas para 'n_registros'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["n_registros"], method="linear")
print(f"Faltantes antes: {df["n_registros"].isna().sum()} | "
      f"después: {df_clean["n_registros"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["n_registros"]

models = {
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: sistemas_informacion_ambiental -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Sistemas de información

- **Metadata-driven:** énfasis en calidad de datos (completeness, freshness, lineage) — no modelado predictivo.
- **Indicadores de cobertura:** % registros con metadatos completos, % datasets actualizados, MTBF de pipelines.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Sistemas de Información Ambiental (`sistemas_informacion_ambiental`)
- **Variable analizada:** `n_registros` ()
- **Modelos ejecutados:** RANDOM_FOREST, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/sistemas_informacion_ambiental.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.